In [1]:
import scipy.io as sio
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import urllib.request
import os

print("Loading SVHN dataset...")

# Create directory
os.makedirs('/tmp/svhn', exist_ok=True)

# Download if not exists
train_file = '/tmp/svhn/train_32x32.mat'
test_file = '/tmp/svhn/test_32x32.mat'

if not os.path.exists(train_file):
    print("Downloading train set...")
    urllib.request.urlretrieve(
        'http://ufldl.stanford.edu/housenumbers/train_32x32.mat',
        train_file
    )
    print("✓ Train downloaded")

if not os.path.exists(test_file):
    print("Downloading test set...")
    urllib.request.urlretrieve(
        'http://ufldl.stanford.edu/housenumbers/test_32x32.mat',
        test_file
    )
    print("✓ Test downloaded")

# Load .mat files
print("Loading data from .mat files...")
train_data = sio.loadmat(train_file)
test_data = sio.loadmat(test_file)

# Extract arrays
X_train = train_data['X']
y_train = train_data['y']
X_test = test_data['X']
y_test = test_data['y']

# Transpose: (H, W, C, N) -> (N, H, W, C)
X_train = np.transpose(X_train, (3, 0, 1, 2))
X_test = np.transpose(X_test, (3, 0, 1, 2))

# Fix labels: 10 -> 0
y_train[y_train == 10] = 0
y_test[y_test == 10] = 0

print(f"✓ Train: {X_train.shape}")
print(f"✓ Test: {X_test.shape}")

# Normalize
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Categorical
y_cat_train = to_categorical(y_train, 10)
y_cat_test = to_categorical(y_test, 10)

print("✓ SVHN loaded successfully!")

2026-03-22 01:20:10.727788: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774142410.932488      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774142410.987553      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774142411.437728      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774142411.437781      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774142411.437784      24 computation_placer.cc:177] computation placer alr

Loading SVHN dataset...
✓ Train downloaded
✓ Test downloaded
Loading data from .mat files...
✓ Train: (73257, 32, 32, 3)
✓ Test: (26032, 32, 32, 3)
✓ SVHN loaded successfully!


In [2]:
!pip uninstall codecarbon -y
!pip install codecarbon==2.3.4 --quiet


import os
# Smell 3 injection: Library Path Mismatch
os.environ["LD_LIBRARY_PATH"] = "/tmp:/doesnotexist:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from codecarbon import EmissionsTracker
import psutil
import subprocess
import time
import gc

# Configuration
SEED = 42
BATCH_SIZE = 32
EPOCHS = 50
NUM_RUNS = 10
VARIANT = "smell 3 Library Mismatch_CNN_svhn"
CSV_FILE = "smell 3 Library Mismatch_CNN_svhn_results.csv"

print("="*70)
print("M3 - CNN ON SVHN - smell 3 Library Mismatch ")
print("="*70)

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✓ GPU: {len(gpus)} device(s)")
else:
    print("⚠ No GPU")
print()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.6/181.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.0 MB/s eta 0:00:00
M3 - CNN ON SVHN - smell 3 Library Mismatch 
✓ GPU: 1 device(s)



In [3]:
def build_model():
    """Build 3-block CNN"""
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), input_shape=(32, 32, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.25),
        
        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.25),
        
        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.25),
        
        # Dense
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.25),
        Dense(10, activation='softmax')
    ])
    return model

print("✓ Model ready")

✓ Model ready


In [4]:
class MemoryTracker:
    """Track memory using nvidia-smi"""
    def __init__(self, phase_name):
        self.phase_name = phase_name
        self.process = psutil.Process()
        self.start_cpu = 0
        self.start_gpu = 0
        self.peak_cpu = 0
        self.peak_gpu = 0
        
    def get_gpu_memory(self):
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                timeout=5
            )
            if result.returncode == 0:
                memory_str = result.stdout.decode('utf-8').strip().split('\n')[0]
                return float(memory_str)
            return 0
        except:
            return 0
            
    def start(self):
        self.start_cpu = self.process.memory_info().rss / (1024**2)
        self.start_gpu = self.get_gpu_memory()
            
    def stop(self):
        end_cpu = self.process.memory_info().rss / (1024**2)
        end_gpu = self.get_gpu_memory()
        self.peak_cpu = end_cpu - self.start_cpu
        self.peak_gpu = end_gpu - self.start_gpu
        return self.peak_cpu, self.peak_gpu

print("✓ MemoryTracker ready")

✓ MemoryTracker ready


In [5]:
all_results = []

for run_id in range(1, NUM_RUNS + 1):
    print(f"\n{'='*70}")
    print(f"RUN {run_id}/{NUM_RUNS}")
    print(f"{'='*70}\n")
    
    try:
        tf.keras.backend.clear_session()
        gc.collect()
        
        np.random.seed(SEED + run_id)
        tf.random.set_seed(SEED + run_id)
        
        # Build model
        print("Building model...")
        model = build_model()
        model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy',
                    tf.keras.metrics.Precision(name='precision'),
                    tf.keras.metrics.Recall(name='recall')]
        )

        # Early Stopping
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
)
        
        # CRITICAL: Create TF Dataset for GPU efficiency
        train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_cat_train))
        train_ds = train_ds.shuffle(10000, seed=SEED + run_id).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        
        test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_cat_test))
        test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        
        # Training
        print(f"\nTraining ({EPOCHS} epochs)...")
        
        mem_tracker_train = MemoryTracker("train")
        mem_tracker_train.start()
        
        tracker_train = EmissionsTracker(
            project_name=f"{VARIANT}_train_{run_id}",
            output_dir=".",
            output_file="codecarbon_train.csv",
            log_level="error"
        )
        tracker_train.start()
        
        train_start = time.time()
        
        history = model.fit(
            train_ds,  # TF Dataset!
            epochs=EPOCHS,
            validation_data=test_ds,
             callbacks=[early_stop],
            verbose=2
        )
        
        train_time = time.time() - train_start
        train_emissions = tracker_train.stop()
        train_energy = getattr(tracker_train.final_emissions_data, "energy_consumed", None)
        
        cpu_train, gpu_train = mem_tracker_train.stop()
        
        print(f"✓ Training: {train_time:.1f}s")
        print(f"  CPU: +{cpu_train:.1f} MB | GPU: +{gpu_train:.1f} MB")
        
        # Evaluation
        print("\nEvaluating...")
        
        mem_tracker_eval = MemoryTracker("eval")
        mem_tracker_eval.start()
        
        tracker_eval = EmissionsTracker(
            project_name=f"{VARIANT}_eval_{run_id}",
            output_dir=".",
            output_file="codecarbon_eval.csv",
            log_level="error"
        )
        tracker_eval.start()
        
        eval_start = time.time()
        results_eval = model.evaluate(test_ds, verbose=0)
        eval_time = time.time() - eval_start
        
        eval_emissions = tracker_eval.stop()
        eval_energy = getattr(tracker_eval.final_emissions_data, "energy_consumed", None)
        
        cpu_eval, gpu_eval = mem_tracker_eval.stop()
        
        test_acc = results_eval[1]
        test_prec = results_eval[2]
        test_rec = results_eval[3]
        
        print(f"✓ Acc: {test_acc:.4f} | Prec: {test_prec:.4f} | Rec: {test_rec:.4f}")
        
        # Save
        results = {
            'variant': VARIANT,
            'run_id': run_id,
            'train_time_sec': round(train_time, 4),
            'eval_time_sec': round(eval_time, 4),
            'cpu_peak_train_mb': round(cpu_train, 2),
            'gpu_peak_train_mb': round(gpu_train, 2) if gpu_train > 0 else "NA",
            'cpu_peak_eval_mb': round(cpu_eval, 2),
            'gpu_peak_eval_mb': round(gpu_eval, 2) if gpu_eval > 0 else "NA",
            'train_energy_kwh': round(train_energy, 6) if train_energy else "NA",
            'train_co2_kg': round(train_emissions, 6) if train_emissions else "NA",
            'eval_energy_kwh': round(eval_energy, 6) if eval_energy else "NA",
            'eval_co2_kg': round(eval_emissions, 6) if eval_emissions else "NA",
            'test_accuracy': round(test_acc, 4),
            'test_precision': round(test_prec, 4),
            'test_recall': round(test_rec, 4),
        }
        
        all_results.append(results)
        df = pd.DataFrame(all_results)
        df.to_csv(CSV_FILE, index=False)
        
        print(f"✓ Saved to {CSV_FILE}\n")
        
        del model
        gc.collect()
        
    except Exception as e:
        print(f"✗ ERROR: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*70)
print("COMPLETE")
print("="*70)
if all_results:
    df = pd.DataFrame(all_results)
    print(f"✓ {len(all_results)}/{NUM_RUNS} runs")


RUN 1/10

Building model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1774142474.287899      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5



Training (50 epochs)...
Epoch 1/50


I0000 00:00:1774142489.442538      78 service.cc:152] XLA service 0x7dcc84003c30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774142489.442573      78 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1774142490.289041      78 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774142497.009814      78 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2290/2290 - 34s - 15ms/step - accuracy: 0.6811 - loss: 0.9553 - precision: 0.8744 - recall: 0.5902 - val_accuracy: 0.8460 - val_loss: 0.5021 - val_precision: 0.9136 - val_recall: 0.7880
Epoch 2/50
2290/2290 - 14s - 6ms/step - accuracy: 0.8780 - loss: 0.4209 - precision: 0.9242 - recall: 0.8399 - val_accuracy: 0.9043 - val_loss: 0.3311 - val_precision: 0.9402 - val_recall: 0.8721
Epoch 3/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9023 - loss: 0.3406 - precision: 0.9394 - recall: 0.8731 - val_accuracy: 0.9226 - val_loss: 0.2743 - val_precision: 0.9494 - val_recall: 0.9010
Epoch 4/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9143 - loss: 0.2996 - precision: 0.9460 - recall: 0.8902 - val_accuracy: 0.9289 - val_loss: 0.2569 - val_precision: 0.9534 - val_recall: 0.9082
Epoch 5/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9234 - loss: 0.2707 - precision: 0.9504 - recall: 0.9010 - val_accuracy: 0.9363 - val_loss: 0.2420 - val_precision: 0.9566 - val_recall: 0.9174
Epoch 6/50
2290/2290 - 14s - 6

...


Training (50 epochs)...
Epoch 1/50
2290/2290 - 32s - 14ms/step - accuracy: 0.6836 - loss: 0.9573 - precision: 0.8734 - recall: 0.5915 - val_accuracy: 0.8846 - val_loss: 0.3925 - val_precision: 0.9296 - val_recall: 0.8505
Epoch 2/50
2290/2290 - 14s - 6ms/step - accuracy: 0.8780 - loss: 0.4195 - precision: 0.9260 - recall: 0.8401 - val_accuracy: 0.9176 - val_loss: 0.2989 - val_precision: 0.9442 - val_recall: 0.8949
Epoch 3/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9013 - loss: 0.3448 - precision: 0.9390 - recall: 0.8728 - val_accuracy: 0.9311 - val_loss: 0.2530 - val_precision: 0.9544 - val_recall: 0.9113
Epoch 4/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9144 - loss: 0.3031 - precision: 0.9458 - recall: 0.8898 - val_accuracy: 0.9319 - val_loss: 0.2469 - val_precision: 0.9556 - val_recall: 0.9143
Epoch 5/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9251 - loss: 0.2667 - precision: 0.9518 - recall: 0.9029 - val_accuracy: 0.9414 - val_loss: 0.2277 - val_precision: 0.9583 - val_recall: 0

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training (50 epochs)...
Epoch 1/50
2290/2290 - 32s - 14ms/step - accuracy: 0.6857 - loss: 0.9565 - precision: 0.8719 - recall: 0.5930 - val_accuracy: 0.8866 - val_loss: 0.3918 - val_precision: 0.9294 - val_recall: 0.8505
Epoch 2/50
2290/2290 - 14s - 6ms/step - accuracy: 0.8778 - loss: 0.4198 - precision: 0.9252 - recall: 0.8406 - val_accuracy: 0.9015 - val_loss: 0.3383 - val_precision: 0.9409 - val_recall: 0.8675
Epoch 3/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9035 - loss: 0.3425 - precision: 0.9392 - recall: 0.8746 - val_accuracy: 0.9275 - val_loss: 0.2595 - val_precision: 0.9509 - val_recall: 0.9108
Epoch 4/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9154 - loss: 0.2987 - precision: 0.9472 - recall: 0.8913 - val_accuracy: 0.9365 - val_loss: 0.2363 - val_precision: 0.9577 - val_recall: 0.9179
Epoch 5/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9258 - loss: 0.2656 - precision: 0.9528 - recall: 0.9045 - val_accuracy: 0.9360 - val_loss: 0.2319 - val_precision: 0.9578 - val_recall: 0

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training (50 epochs)...
Epoch 1/50
2290/2290 - 32s - 14ms/step - accuracy: 0.6493 - loss: 1.0298 - precision: 0.8660 - recall: 0.5516 - val_accuracy: 0.8448 - val_loss: 0.5053 - val_precision: 0.9108 - val_recall: 0.7876
Epoch 2/50
2290/2290 - 14s - 6ms/step - accuracy: 0.8745 - loss: 0.4275 - precision: 0.9225 - recall: 0.8359 - val_accuracy: 0.9056 - val_loss: 0.3316 - val_precision: 0.9437 - val_recall: 0.8689
Epoch 3/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9012 - loss: 0.3478 - precision: 0.9389 - recall: 0.8717 - val_accuracy: 0.9265 - val_loss: 0.2670 - val_precision: 0.9481 - val_recall: 0.9086
Epoch 4/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9152 - loss: 0.3003 - precision: 0.9467 - recall: 0.8906 - val_accuracy: 0.8684 - val_loss: 0.4391 - val_precision: 0.9120 - val_recall: 0.8293
Epoch 5/50
2290/2290 - 14s - 6ms/step - accuracy: 0.9231 - loss: 0.2709 - precision: 0.9507 - recall: 0.9011 - val_accuracy: 0.9348 - val_loss: 0.2404 - val_precision: 0.9551 - val_recall: 0